# Sesión 05 — Árboles de regresión y ajuste de hiperparámetros

**Bloque:** 2 · Regresión

## Objetivos

Al terminar esta sesión deberías ser capaz de:

- Explicar cómo un árbol de regresión decide dónde partir los datos, y qué predice exactamente una hoja.
- Leer un árbol dibujado con `plot_tree` y traducir una rama a una frase en castellano.
- Diagnosticar sobreajuste y subajuste mirando juntas las curvas de error de entrenamiento y de prueba, en vez de una sola cifra.
- Ajustar `max_depth`, `min_samples_leaf` y `min_samples_split` a mano primero y con `GridSearchCV` después, sabiendo qué hace la herramienta por dentro.
- Interpretar `feature_importances_` y decir qué **no** significa esa cifra.
- Decidir si un problema pide un árbol o un modelo lineal, con un argumento que no sea "el árbol es más moderno".

## Qué necesitas de antes

- **S03**: `train_test_split`, RMSE y R², y el modelo lineal sobre el dataset de seguros — hoy volvemos a él.
- **S04**: validación cruzada. `GridSearchCV` es validación cruzada por dentro, repetida para cada combinación de hiperparámetros.
- Del ejercicio 3 de S04 necesitas **tus RMSE de Lasso y Ridge sobre California Housing**. Si no los tienes a mano, vuelve a ejecutarlo: hoy se comparan.

## 1. ¿Qué es un árbol de decisión?

Un árbol de decisión predice haciendo una **secuencia de preguntas de sí o no** sobre las variables. "¿El ingreso medio del distrito es menor que 5,03? Si sí, ¿la latitud es menor que 34,4?"... y así hasta llegar a una respuesta.

Su vocabulario:

- **Nodo raíz**: la primera pregunta, la que parte todo el conjunto.
- **Nodo interno**: cualquier pregunta intermedia sobre una variable.
- **Hoja**: un nodo final que ya no se parte. Aquí está la predicción.

**Lo que hay dentro de una hoja es la idea clave de la sesión.** En regresión, una hoja no guarda una fórmula ni una recta: guarda **un solo número**, la media de los valores de entrenamiento que cayeron ahí. Si a una hoja llegaron 30 casas cuyo precio medio era 250.000 $, el árbol predecirá 250.000 $ para cualquier casa nueva que caiga en esa hoja, sea como sea.

Eso tiene dos consecuencias que conviene ver desde el principio: el árbol predice **a escalones**, no de forma suave; y **nunca predice fuera del rango que vio entrenando**. Si la casa más cara del entrenamiento costó 500.000 $, el árbol jamás dirá 600.000 $.

**¿Y cómo decide dónde cortar?** Prueba todos los cortes posibles (cada variable, cada valor umbral) y se queda con el que deja los dos grupos resultantes **más homogéneos** entre sí. En regresión, "homogéneo" significa con menos error cuadrático medio dentro de cada grupo. Repite el proceso en cada nodo hasta que le digan que pare — y decirle cuándo parar es exactamente el trabajo de hoy.

**Qué problema resuelve:** predecir cuando la relación no es una línea recta, o cuando el efecto de una variable depende del valor de otra.

**Cuándo elegirlo:** cuando sospechas relaciones curvas o por tramos ("el precio sube con la antigüedad hasta los 30 años y luego baja"), cuando hay **interacciones** entre variables, o cuando necesitas un modelo que alguien no técnico pueda seguir con el dedo.

**Cuándo NO:** cuando la relación es de verdad lineal — ahí un árbol la aproxima a escalones y sale perdiendo. Tampoco si necesitas extrapolar más allá del rango visto, ni si necesitas un modelo estable: cambiar unas pocas filas de entrenamiento puede reorganizar el árbol entero.

**Qué asume:** muy poco, y esa es su gracia. No asume linealidad, ni normalidad, ni independencia entre variables. **No necesita escalado**: como los cortes son umbrales sobre cada variable por separado, da igual que una vaya de 0 a 1 y otra de 0 a 1.000.000. Después de la disciplina de escalado de S04, esto sorprende.

In [ ]:
# Configuración para todo el notebook
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error

sns.set_theme(style="whitegrid")

# Cargamos el dataset de vivienda de California
url = 'https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv'
df = pd.read_csv(url)

# Para las visualizaciones usamos solo 2 variables y 100 filas: así el árbol cabe en pantalla
X_viz = df[['median_income', 'housing_median_age']].head(100)
y_viz = df['median_house_value'].head(100)

print(f"Dataset completo: {df.shape[0]} distritos")
print(f"Subconjunto para dibujar: {X_viz.shape[0]} filas y {X_viz.shape[1]} variables")

### Demostración: un árbol que cabe en pantalla

Lo ejecutamos juntos. Empezamos por un árbol de profundidad 2 sobre 100 distritos y dos variables, para poder leerlo entero.

En cada nodo, `plot_tree` muestra la pregunta, el error cuadrático medio del grupo, cuántas muestras hay y `value`, que es **la predicción de ese nodo**. Fíjate en el `value` de las hojas: ese es el número que devolverá el modelo.

In [ ]:
arbol_prof2 = DecisionTreeRegressor(max_depth=2, random_state=42).fit(X_viz, y_viz)

plt.figure(figsize=(18, 10))
plot_tree(arbol_prof2, feature_names=X_viz.columns, filled=True, rounded=True)
plt.title("Árbol con max_depth = 2", fontsize=18)
plt.show()

print(f"Hojas: {arbol_prof2.get_n_leaves()} | Profundidad real: {arbol_prof2.get_depth()}")

Ahora el mismo árbol con un nivel más. Compara los dos dibujos: `max_depth=3` tiene una capa más de preguntas, hace distinciones más finas... y por eso mismo corre más riesgo de aprenderse el ruido.

In [ ]:
arbol_prof3 = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_viz, y_viz)

plt.figure(figsize=(22, 12))
plot_tree(arbol_prof3, feature_names=X_viz.columns, filled=True, rounded=True)
plt.title("Árbol con max_depth = 3", fontsize=18)
plt.show()

print(f"Hojas: {arbol_prof3.get_n_leaves()} | Profundidad real: {arbol_prof3.get_depth()}")

# En texto se lee mejor que en dibujo cuando el árbol crece
print("\nEl mismo árbol, en texto:")
print(export_text(arbol_prof3, feature_names=list(X_viz.columns), max_depth=2))

## 2. Los tornillos del árbol: los hiperparámetros

Un árbol sin límites crece hasta que cada hoja tiene una sola muestra: memoriza el entrenamiento y no sirve para nada nuevo. Los **hiperparámetros** son las reglas que le ponemos para que pare antes. Los tres principales:

- **`max_depth`** — cuántos niveles de preguntas puede encadenar. Es el más directo y el que se toca primero.
- **`min_samples_split`** — "no partas un nodo si no tiene al menos N muestras". Mira el nodo **antes** de partirlo.
- **`min_samples_leaf`** — "ninguna hoja puede quedarse con menos de N muestras". Mira los hijos **resultantes**, y por eso es el que de verdad controla sobre cuántos casos se calcula cada predicción.

La diferencia entre los dos últimos es sutil y se confunde siempre. `min_samples_split=50` permite una partición que deje una hoja con 2 muestras, mientras el nodo padre tuviera 50. `min_samples_leaf=20` lo prohíbe directamente.

Los tres hacen lo mismo por caminos distintos: **impedir que el árbol siga preguntando cuando ya se está inventando cosas.**

In [ ]:
# min_samples_split = 50: un nodo con menos de 50 muestras ya no se parte
arbol_split = DecisionTreeRegressor(max_depth=4, min_samples_split=50, random_state=42).fit(X_viz, y_viz)

plt.figure(figsize=(15, 8))
plot_tree(arbol_split, feature_names=X_viz.columns, filled=True, rounded=True)
plt.title("max_depth=4 pero min_samples_split=50", fontsize=16)
plt.show()

print(f"Hojas: {arbol_split.get_n_leaves()} | Profundidad real: {arbol_split.get_depth()}")
print("Le dejamos crecer hasta 4 niveles y no ha llegado: se quedó sin muestras antes.")

In [ ]:
# min_samples_leaf = 20: ninguna hoja puede basarse en menos de 20 casas
arbol_leaf = DecisionTreeRegressor(max_depth=4, min_samples_leaf=20, random_state=42).fit(X_viz, y_viz)

plt.figure(figsize=(15, 8))
plot_tree(arbol_leaf, feature_names=X_viz.columns, filled=True, rounded=True)
plt.title("max_depth=4 pero min_samples_leaf=20", fontsize=16)
plt.show()

print(f"Hojas: {arbol_leaf.get_n_leaves()} | Profundidad real: {arbol_leaf.get_depth()}")
print("Cada predicción final es ahora la media de al menos 20 distritos, no de dos o tres.")

---
### Preguntas de recap — El árbol y sus tornillos

1. Una hoja de tu árbol predice 250.000 $. ¿De dónde sale ese número?
2. ¿Por qué un árbol nunca predice un valor mayor que el máximo que vio entrenando?
3. ¿En qué se diferencian `min_samples_split` y `min_samples_leaf`?
4. ¿Por qué un árbol no necesita que escales las variables, si en S04 era obligatorio?
5. Si pones `max_depth=1`, ¿cuántas predicciones distintas puede dar el modelo como mucho?

---

## 3. El diagnóstico: sobreajuste contra subajuste

Ya sabemos controlar la complejidad. Falta saber **cuánta hace falta**, y para eso hay que mirar dos errores a la vez, no uno.

- **Árbol demasiado profundo**: memoriza el ruido del entrenamiento. Error de entrenamiento casi cero, error de prueba malo y **empeorando**. Es **sobreajuste**.
- **Árbol demasiado podado**: no captura ni lo evidente. Error alto en los dos sitios. Es **subajuste**.

La herramienta de diagnóstico es dibujar las dos curvas de error frente a la complejidad. El sobreajuste no se ve en una cifra: se ve en la **separación** entre las dos curvas.

> **Aviso importante, y no es un detalle.** La curva que vamos a dibujar usa el conjunto de **test** para medir. Sirve para *ver* el fenómeno, que es de lo que va esta sección. Pero si además eliges el `max_depth` mirando dónde está el mínimo de esa curva, has usado el test para tomar una decisión de modelado — y el error que reportes después ya no es una estimación honesta de nada. La forma correcta es decidir con validación cruzada dentro del entrenamiento, que es exactamente lo que hará `GridSearchCV` en el ejercicio 2. Guárdate esta observación: hay una reflexión sobre ella.

In [ ]:
# Preparamos el dataset completo (los árboles no necesitan escalado)
df_numeric = df.select_dtypes(include=np.number).copy()
for col in df_numeric.columns:
    if df_numeric[col].isnull().any():
        df_numeric[col] = df_numeric[col].fillna(df_numeric[col].median())

X = df_numeric.drop('median_house_value', axis=1)
y = df_numeric['median_house_value']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Un árbol por cada profundidad, midiendo el error en los dos conjuntos
profundidades = range(1, 21)
rmse_train, rmse_test = [], []

for d in profundidades:
    arbol = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    rmse_train.append(np.sqrt(mean_squared_error(y_train, arbol.predict(X_train))))
    rmse_test.append(np.sqrt(mean_squared_error(y_test, arbol.predict(X_test))))

plt.figure(figsize=(12, 7))
plt.plot(profundidades, rmse_train, 'o-', label='Error en entrenamiento')
plt.plot(profundidades, rmse_test, 'o-', label='Error en prueba')
plt.xlabel('Profundidad del árbol (max_depth)')
plt.ylabel('RMSE ($)')
plt.title('Error del árbol frente a su profundidad', fontsize=16)
plt.xticks(list(profundidades))
plt.legend()
plt.grid(True)
plt.show()

mejor = list(profundidades)[int(np.argmin(rmse_test))]
print(f"Mínimo del error de prueba en max_depth={mejor}: ${min(rmse_test):,.0f}")
print(f"En esa profundidad, el error de entrenamiento es ${rmse_train[mejor - 1]:,.0f}")
print(f"Con max_depth=20: entrenamiento ${rmse_train[19]:,.0f} | prueba ${rmse_test[19]:,.0f}")

**Cómo se lee este gráfico.**

La curva de entrenamiento **baja siempre**. Siempre. Añadir profundidad nunca empeora el ajuste sobre los datos que el modelo ya ha visto: con `max_depth=20` el error de entrenamiento cae por debajo de 6.500 $, prácticamente memorización.

La de prueba baja, toca fondo y **vuelve a subir**. Ese punto de inflexión es la frontera entre aprender y memorizar. A la izquierda hay subajuste: las dos curvas están altas y juntas. A la derecha hay sobreajuste: se abren en abanico, y esa distancia entre ambas *es* el sobreajuste dibujado.

Lo que hay que llevarse: **un error de entrenamiento excelente no es una buena noticia por sí solo.** Aislado, no distingue un modelo que ha aprendido de uno que se ha aprendido de memoria la respuesta.

---
### Preguntas de recap — Sobreajuste

1. ¿Por qué la curva de error de entrenamiento nunca sube al aumentar la profundidad?
2. Tienes un modelo con RMSE de 6.000 $ en entrenamiento y 69.000 $ en prueba. ¿Qué le pasa?
3. ¿Y uno con 95.000 $ en entrenamiento y 97.000 $ en prueba?
4. ¿Qué tiene de malo elegir `max_depth` mirando el mínimo de la curva de prueba?
5. Si solo pudieras mirar una de las dos curvas, ¿cuál te dice más? ¿Por qué?

---

---
## Ejercicio 1 — El árbol vuelve al dataset de seguros

**Contexto.** La aseguradora de S03 no quedó contenta. Tu modelo lineal explicaba bien a la mayoría de clientes, pero se equivocaba sistemáticamente con los más caros: había una franja entera de asegurados que el modelo infravaloraba. Te dan otra oportunidad con la misma pregunta —predecir `charges`— y libertad para cambiar de modelo.

Vas a probar un árbol. La sospecha es que el efecto de fumar **no se suma** al del índice de masa corporal, sino que **se multiplica**: fumar con un BMI alto no cuesta lo mismo que fumar con un BMI normal. Un modelo lineal no puede representar eso salvo que se lo escribas a mano. Un árbol lo descubre solo.

**Tu tarea.**
1. Carga el dataset de seguros desde `https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv` y codifica `sex`, `smoker` y `region` con `pd.get_dummies(..., drop_first=True)`. Separa en train y test con `random_state=42`.
2. Construye la curva de error de entrenamiento y de prueba para `max_depth` de 1 a 15, igual que en la demostración.
3. Elige una profundidad a partir del gráfico y **escribe la justificación** en la celda de reflexión. No hay una única respuesta buena; lo que se evalúa es el argumento.
4. Entrena tu árbol con esa profundidad y da su RMSE y su R² en test.
5. Dibuja `feature_importances_` en un gráfico de barras.

**Criterio de que lo has hecho bien:** el punto de inflexión de este dataset está **mucho antes** que el de California — si te sale en la misma zona, revisa el código. La variable dominante debería coincidir con la que tenía el coeficiente más grande en S03, y varias variables deberían quedarse con importancia exactamente cero. El R² tiene que superar al del modelo lineal de S03.

In [ ]:
# TODO: completa esta celda
# La curva se construye igual que en la demostración: un bucle que entrena un árbol por profundidad y guarda los dos RMSE.
# Para `feature_importances_` necesitas emparejar el array con `X_seg.columns`, en ese mismo orden.


### Mis reflexiones

*(Escribe aquí tu respuesta antes de seguir)*

**Mi elección de `max_depth`:** anota la profundidad que has elegido y por qué, describiendo qué viste en el gráfico. Si has elegido una distinta al mínimo exacto de la curva, explica el criterio.

- La profundidad que va bien aquí no es la que iba bien en California. ¿Qué tiene este dataset que justifique la diferencia? Da al menos una razón concreta mirando sus dimensiones.
- `sex` y las tres variables de `region` tienen importancia exactamente 0. ¿Dirías que la región donde vives no afecta a lo que pagas de seguro? Explica qué significa exactamente ese 0.
- El árbol mejora el R² del modelo lineal de S03 sobre los mismos datos y sin variables nuevas. ¿Qué está capturando el árbol que el modelo lineal no podía? Intenta describirlo como una frase sobre asegurados, no sobre matemáticas.
- Elegiste la profundidad mirando la curva de error de **prueba**. ¿Qué le responderías a alguien que te dice que ese RMSE final ya no es una estimación honesta?

---
## Ejercicio 2 — Dejar de ajustar a ojo: `GridSearchCV`

**Contexto.** Volvemos a California. Hasta ahora has tocado un tornillo cada vez y a ojo, mirando un gráfico. Eso no escala: son tres hiperparámetros y sus efectos se solapan, así que la mejor profundidad depende de qué `min_samples_leaf` tengas puesto. Probar las combinaciones a mano es inviable.

`GridSearchCV` hace justo eso: recorre todas las combinaciones de una rejilla y evalúa cada una **con validación cruzada dentro del entrenamiento** — la de S04. Por eso su elección sí es honesta: nunca mira el test.

El equipo de dirección quiere una recomendación con números, y ya tienes con qué compararla: en el ejercicio 3 de S04 mediste Lasso y Ridge sobre **este mismo dataset**.

**Tu tarea.**
1. Define una rejilla con `max_depth` en `[7, 9, 11, 13]`, `min_samples_leaf` en `[5, 10, 15]` y `min_samples_split` en `[10, 20, 30]`. Calcula cuántos ajustes de modelo va a suponer eso con `cv=5` antes de lanzarlo.
2. Lanza `GridSearchCV` con `scoring='neg_mean_squared_error'` y `n_jobs=-1`. Muestra `best_params_` y el RMSE de validación cruzada del ganador.
3. Evalúa el mejor modelo en test y compáralo con el árbol sin tunear de la demostración.
4. Monta una tabla final con cuatro filas: el árbol simple (`max_depth=3`), el árbol de la demo, tu árbol optimizado, y **tu Lasso de California del ejercicio 3 de S04**.
5. **Mira dónde ha caído `best_params_` dentro de la rejilla que le diste.** ¿Está alguno de los valores ganadores en el extremo de su lista? Si es así, di qué habría que hacer y por qué.

**Criterio de que lo has hecho bien:** el árbol optimizado debe mejorar tanto al árbol sin tunear como a los modelos lineales de S04, y por un margen que se ve a simple vista. Y el punto 5 tiene respuesta: hay algo raro en `best_params_`, y encontrarlo vale más que el propio RMSE.

In [ ]:
# TODO: completa esta celda
# El número de ajustes no es el de combinaciones: cada una se entrena una vez por fold.
# `best_estimator_` ya viene reentrenado con todos los datos de entrenamiento, no hace falta volver a llamar a `fit`.


In [ ]:
# TODO: completa esta celda
# Para la tabla te vale un DataFrame de cuatro filas; no hace falta reentrenar nada que ya tengas calculado.
# Para el punto 5, compara el valor ganador con el primero y el último de su lista en la rejilla.


### Mis reflexiones

*(Escribe aquí tu respuesta)*

- `GridSearchCV` eligió el `max_depth` más grande que le ofreciste. ¿Qué conclusión sacas y qué harías a continuación?
- La curva de la demo decía que a partir de `max_depth=10` el modelo empeoraba, y sin embargo el ganador tiene profundidad 13. ¿Se contradicen? Explica por qué no.
- El árbol mejora al Lasso de S04 en más de un 10 % sobre el mismo dataset. ¿Qué tienen los datos de California que explique esa diferencia? Mira las importancias antes de responder.
- Tienes que recomendar un modelo al comité de dirección. Elige, y di explícitamente qué sacrificas con tu elección.
- `GridSearchCV` con esta rejilla son 180 ajustes. Si añadieras dos hiperparámetros más con cuatro valores cada uno, ¿cuántos serían? ¿Sigue siendo viable? ¿Qué harías si no lo fuera?

---
## Ejercicio 3 — Donde el árbol pierde

**Contexto.** Vuelves a Ames Housing, el dataset de S04: 1.460 viviendas y 36 variables numéricas. Allí entrenaste modelos lineales y anotaste sus RMSE. Ahora tienes una herramienta nueva y la confianza de haber ganado en California por un 14 %.

Tu jefe te pide lo obvio: *"pásale un árbol a Ames también"*.

**Tu tarea.**
1. Carga Ames igual que en S04 (`fetch_openml(data_id=42165)`), quédate con las numéricas, quita `Id` e imputa los nulos con la mediana. No hace falta escalar: explica en la reflexión por qué.
2. Construye la curva de error de entrenamiento y prueba para `max_depth` de 1 a 20, y anota en qué profundidad toca fondo la de prueba.
3. Lanza `GridSearchCV` con la **misma rejilla** del ejercicio 2 y evalúa el ganador en test.
4. Monta una tabla comparando: el mejor árbol de la curva, el árbol de `GridSearchCV`, y **tus RMSE de S04 para `LinearRegression`, Lasso y Ridge sobre Ames**.
5. Mira `feature_importances_` y cuenta cuántas variables se quedan en cero.

**Criterio de que lo has hecho bien:** aquí el árbol **no gana**. Los modelos lineales de S04 quedan por delante. Si te sale que el árbol arrasa, revisa que estás comparando RMSE de test contra RMSE de test.

Y hay un segundo resultado incómodo: el árbol que sale de `GridSearchCV` es **peor** que el mejor árbol de tu curva del punto 2. Eso tiene una explicación, no es un error de código, y encontrarla es la parte importante del ejercicio.

**Entrega:** el notebook con las celdas completadas y las reflexiones escritas.

In [ ]:
# TODO: completa esta celda
# La preparación de Ames es idéntica a la de S04: puedes copiar aquellas celdas, quitando el escalado y el ruido.
# Para el punto 4, la comparación solo vale si todos los RMSE se miden sobre el mismo conjunto de test y con el mismo `random_state`.


### Mis reflexiones finales

*(Escribe aquí tu respuesta)*

- El árbol ganó en California por un 14 % y pierde en Ames. Explica por qué mirando las importancias de los dos modelos. No vale "porque los datos son distintos": di **qué** es distinto.
- `GridSearchCV` te ha dado un árbol peor que el que elegiste tú mirando el gráfico. ¿Significa que mirar el gráfico funciona mejor? Piensa en **cómo** se eligió cada uno antes de responder.
- En este ejercicio no has escalado nada, y en S04 era obligatorio. Explica en una frase por qué el árbol no lo necesita.
- El árbol se apoya en 2 variables para el 88 % de su decisión y deja 16 en cero. Lasso, en S04, repartía entre 23. Si tuvieras que explicarle a un cliente "qué mira el modelo", ¿cuál de las dos respuestas te parece más honesta? ¿Y más útil?
- Has cerrado el bloque de regresión con cuatro familias de modelo: lineal, regularizada, y árbol. Escribe la regla que usarías para elegir entre ellas ante un problema nuevo, antes de entrenar nada.
- Tu árbol nunca predecirá un precio superior al más caro que vio entrenando. Da un caso de negocio donde eso sea inofensivo y otro donde sea inaceptable.

---
## Para cerrar: cuatro preguntas que siempre salen

**¿Por qué el árbol parte por ahí y no por otro sitio?**
Prueba todos los cortes posibles —cada variable, cada valor umbral— y se queda con el que deja los dos grupos más homogéneos. En regresión, "homogéneo" se mide con el error cuadrático medio dentro de cada grupo. El árbol elige la pregunta que produce los dos grupos de precios más parecidos entre sí por dentro.

**¿Y los valores atípicos?**
Los árboles aguantan bien los outliers **en las variables predictoras**: como los cortes son umbrales, un valor extremo en `X` no arrastra nada, al contrario que en una regresión. Un outlier en la **variable objetivo** sí afecta, porque entra en la media de la hoja donde caiga — aunque su efecto se diluye entre el resto de muestras de esa hoja. Cuanto mayor sea `min_samples_leaf`, más se diluye.

**¿Qué gana un árbol frente a una regresión lineal?**
Tres cosas: captura **relaciones no lineales** ("el precio sube con la antigüedad hasta los 30 años y luego baja"); descubre **interacciones** solo, sin que se las escribas (lo que has visto hoy con fumar y el BMI); y **no necesita escalado**. Lo que pierde: estabilidad, capacidad de extrapolar, y esa lectura tan directa que tenían los coeficientes.

**¿Por qué `max_depth=1` va tan mal?**
Un árbol de profundidad 1 se llama *tocón* (*stump*): hace **una sola pregunta** y por tanto solo puede dar **dos** predicciones distintas para todo el dataset. Es el subajuste llevado al extremo. Aun así, no lo descartes del todo: en S06 verás que juntar muchos tocones malos, cada uno corrigiendo los errores del anterior, es exactamente la idea del *boosting*.